# Study RAG pipeline
1. Config  →  2. Chapter metadata  →  3. OCR (cached)  →  4. Pages → chapter documents  →  5. Embedding store (cached)  →  6. Build  →  7. Ask

Run 1–5 every time. Run 6 only when you add books. Run 7 to ask questions.

In [1]:
import os
# Hugging Face Hub cache on Windows is incomplete here (weight file never finished).
# Keep downloads enabled until model_cache has a full snapshot.
os.environ.pop("HF_HUB_OFFLINE", None)
os.environ.pop("TRANSFORMERS_OFFLINE", None)
os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"

import json
import re
from collections import defaultdict
from functools import lru_cache
from pathlib import Path
import pickle

import easyocr
import numpy as np
import torch
import torchvision.transforms.functional as F
from dotenv import load_dotenv
from huggingface_hub import snapshot_download
from pdf2image import convert_from_path
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import pickle

from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

ROOT = Path(r"D:\AI-ML\AI Projects\Study RAG")
PDF_DIR = ROOT / "pdf"                   # pdf/Class_6/BGS.pdf
OCR_CACHE = ROOT / "ocr_cache"           # one json per book (raw OCR text)
EMBED_CACHE = ROOT / "embedding_cache"   # class / book / chapter /
MODEL_CACHE = ROOT / "model_cache"
POPPLER_PATH = r"C:\poppler\Library\bin"
KEYWORDS = ("সূচিপত্র", "সুচিপত্র", "বিষয়সূচি", "CONTENTS", "contents")

OCR_CACHE.mkdir(exist_ok=True)
EMBED_CACHE.mkdir(exist_ok=True)
MODEL_CACHE.mkdir(exist_ok=True)
INDEX_INFOS = json.loads((ROOT / "chapters.json").read_text(encoding="utf-8"))

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=350,
    separators=["\n\n\n\n", "\n\n\n", "\n\n", "\n", "।", " ", ""],
)

c:\Users\Tanvir\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\Tanvir\AppData\Local\Temp\ipykernel_8304\3492996532.py:27: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


## 2. Chapter metadata (from chapters.json)

In [2]:
def get_page_metadatas(class_num, subject, page):
    metas = {}
    if class_num is None or subject is None or page is None or page < 0:
        return metas

    book_section = INDEX_INFOS.get("all_books", {}).get(class_num, {}).get(subject)
    if not book_section:
        return metas

    for chapter in book_section:
        start = chapter.get("starting_page")
        if start is None or start > page:
            continue

        if chapter.get("sub_chapters"):
            for sub in chapter["sub_chapters"]:
                sub_start = sub.get("starting_page")
                if sub_start is not None and sub_start <= page:
                    metas["part"] = chapter.get("chapter_name")
                    metas["chapter"] = sub.get("sub_chapter_name")
                    if sub.get("writer_name"):
                        metas["writer_name"] = sub.get("writer_name")
                    else:
                        metas.pop("writer_name", None)
        else:
            metas["chapter"] = chapter.get("chapter_name")
            metas.pop("part", None)
            metas.pop("writer_name", None)

    return metas

## 3. OCR (GPU, cached — a finished book is never OCR'd again)

In [3]:
_reader = None

def get_reader():
    """Load EasyOCR only when a book actually needs OCR."""
    global _reader
    if _reader is None:
        _reader = easyocr.Reader(["bn", "en"], gpu=torch.cuda.is_available())
    return _reader


def ocr_pdf_to_pages_gpu(pdf_path: Path, poppler_path=POPPLER_PATH, save_every=10):
    class_number = pdf_path.parent.name.lower()
    sub_name = pdf_path.stem.lower()

    cache_file = OCR_CACHE / f"{class_number}_{sub_name}.json"
    partial_file = cache_file.with_suffix(".partial.json")

    if cache_file.exists():
        print(f"OCR loaded from cache: {cache_file.name}")
        texts = json.loads(cache_file.read_text(encoding="utf-8"))
    else:
        # resume a crashed run if a partial file exists
        texts = json.loads(partial_file.read_text(encoding="utf-8")) if partial_file.exists() else []
        if texts:
            print(f"Resuming from page {len(texts) + 1}")

        images = convert_from_path(pdf_path, dpi=300, poppler_path=poppler_path)
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        for i, img in tqdm(enumerate(images), total=len(images), initial=len(texts),
                           desc=f"OCR {sub_name} via {device}"):
            if i < len(texts):
                continue  # already done

            img_tensor = F.to_tensor(img).to(device)
            if img_tensor.shape[0] == 3:  # RGB -> grayscale
                img_tensor = (0.2989 * img_tensor[0] + 0.5870 * img_tensor[1]
                              + 0.1140 * img_tensor[2]).unsqueeze(0)

            min_val, max_val = img_tensor.min(), img_tensor.max()
            if max_val > min_val:  # autocontrast
                img_tensor = (img_tensor - min_val) / (max_val - min_val)

            img_numpy = (img_tensor.squeeze(0).cpu().numpy() * 255).astype("uint8")
            results = get_reader().readtext(img_numpy, paragraph=True)
            texts.append("\n".join(r[1] for r in results))

            if len(texts) % save_every == 0:
                partial_file.write_text(json.dumps(texts, ensure_ascii=False), encoding="utf-8")

        cache_file.write_text(json.dumps(texts, ensure_ascii=False), encoding="utf-8")
        partial_file.unlink(missing_ok=True)

    # page numbering starts at the table-of-contents page; earlier pages stay 0
    pages, tracker, page_num = [], False, 0
    for text in texts:
        if any(k in text for k in KEYWORDS):
            tracker = True
        pages.append({
            "page_number": page_num if tracker else 0,
            "page_content": text,
            "book_name": sub_name,
            "class_number": class_number,
        })
        if tracker:
            page_num += 1
    return pages

## 4. Pages → one document per chapter

In [4]:
def pages_to_documents(book_dicts):
    documents = []
    for item in book_dicts:
        content = item.pop("page_content", "")
        metas = get_page_metadatas(item["class_number"], item["book_name"], item["page_number"])
        documents.append(Document(page_content=content, metadata=item | metas))
    return documents


def merge_documents_by_chapter(documents):
    # key includes class + book, so same-named chapters in different books never mix
    grouped = defaultdict(list)
    for doc in documents:
        m = doc.metadata
        grouped[(m["class_number"], m["book_name"], m.get("chapter", "বই তথ্য"))].append(doc)

    merged = []
    for (cls, book, chapter), docs in grouped.items():
        docs = sorted(docs, key=lambda d: d.metadata["page_number"])
        pages = [d.metadata["page_number"] for d in docs]
        merged.append(Document(
            page_content="\n\n".join(d.page_content for d in docs),
            metadata={"class_number": cls, "book_name": book, "chapter": chapter,
                      "page_start": min(pages), "page_end": max(pages)},
        ))
    return merged

## 5. Embedding store (cached as `embedding_cache/class/book/chapter/`)

This cell is self-contained, so you can move it into `embedding_store.py` unchanged and `import` it from LangGraph.

In [5]:
@lru_cache(maxsize=1)
def get_model():
    """Load E5 from the project cache and make the selected device explicit."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading multilingual-e5-large on {device} ...")
    return SentenceTransformer("intfloat/multilingual-e5-large", cache_folder=str(MODEL_CACHE), device=device)


class E5Embeddings(Embeddings):
    """Adds the E5 'query: ' / 'passage: ' prefixes, so the stored text stays clean."""
    def embed_documents(self, texts):
        batch_size = 16 if torch.cuda.is_available() else 4
        return get_model().encode(["passage: " + t for t in texts],
                                  normalize_embeddings=True, batch_size=batch_size,
                                  show_progress_bar=True).tolist()

    def embed_query(self, text):
        return get_model().encode("query: " + text, normalize_embeddings=True).tolist()


def _path(cls, book, chapter):
    chapter = "".join("_" if c in '<>:"/\\|?*' else c for c in chapter).strip().rstrip(".")
    return EMBED_CACHE / cls / book / chapter


def make_embeddings(chunks, cls, book, chapter, force=False):
    out = _path(cls, book, chapter)
    if (out / "faiss.index").exists() and not force:
        return
    out.mkdir(parents=True, exist_ok=True)

    (out / "chapter_name.txt").write_text(chapter, encoding="utf-8")   # <-- add here

    texts = [c.page_content for c in chunks]
    vecs = E5Embeddings().embed_documents(texts)
    vs = FAISS.from_embeddings(
        text_embeddings=list(zip(texts, vecs)),
        embedding=E5Embeddings(),
        metadatas=[c.metadata for c in chunks],
    )
    vs.save_local(str(out))

    bm25 = BM25Retriever.from_documents(chunks, preprocess_func=_bn_tokenize)
    (out / "bm25.pkl").write_bytes(pickle.dumps(bm25))

# def make_embeddings(chunks, cls, book, chapter, force=False):
#     """Embed one chapter's chunks and cache them. Skips chapters already cached."""
#     out = _path(cls, book, chapter)
#     if (out / "vectors.npy").exists() and not force:
#         return
#     out.mkdir(parents=True, exist_ok=True)
#     vecs = np.array(E5Embeddings().embed_documents([c.page_content for c in chunks]), dtype="float32")
#     np.save(out / "vectors.npy", vecs)
#     (out / "chunks.json").write_text(
#         json.dumps([{"text": c.page_content, "meta": c.metadata} for c in chunks],
#                    ensure_ascii=False), encoding="utf-8")


def get_embeddings(cls, book, chapter):
    out = _path(cls, book, chapter)
    vecs = np.load(out / "vectors.npy")
    chunks = json.loads((out / "chunks.json").read_text(encoding="utf-8"))
    return vecs, chunks


def list_classes():
    return sorted(p.name for p in EMBED_CACHE.iterdir() if p.is_dir())

def list_books(cls):
    return sorted(p.name for p in (EMBED_CACHE / cls).iterdir() if p.is_dir())

def list_chapters(cls, book):
    return sorted(p.name for p in (EMBED_CACHE / cls / book).iterdir() if p.is_dir())


def _bn_tokenize(text):
    return re.sub(r"[।,;:!?\"'()\[\]{}\-–—.]", " ", text).split()


@lru_cache(maxsize=32)  # built once per chapter, reused on every query
def get_hybrid_retriever(cls, book, chapter, k=5, weights=(0.4, 0.6)):
    out = _path(cls, book, chapter)
    vs = FAISS.load_local(str(out), E5Embeddings(), allow_dangerous_deserialization=True)
    bm25 = pickle.loads((out / "bm25.pkl").read_bytes())
    bm25.k = k
    dense = vs.as_retriever(search_kwargs={"k": k})
    return EnsembleRetriever(retrievers=[bm25, dense], weights=list(weights))


In [6]:
print([p.name for p in MODEL_CACHE.iterdir()])
print((Path.home() / ".cache" / "huggingface" / "hub" / "models--intfloat--multilingual-e5-large").exists())

['.locks', 'models--intfloat--multilingual-e5-large']
True


In [7]:
from pathlib import Path
from huggingface_hub import snapshot_download

# Run this once if get_model reports a missing/incomplete model download.
# snapshot_download resumes partial files, so it does not restart from zero.
snapshot_download(
    "intfloat/multilingual-e5-large",
    cache_dir=str(MODEL_CACHE),
    allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt"],  # skips duplicate ONNX/OpenVINO copies
)

Fetching 14 files: 100%|██████████| 14/14 [00:00<00:00, 75670.43it/s]


'D:\\AI-ML\\AI Projects\\Study RAG\\model_cache\\models--intfloat--multilingual-e5-large\\snapshots\\3d7cfbdacd47fdda877c5cd8a79fbcc4f2a574f3'

## 6. Build (run when you add books — finished books and chapters are skipped)

In [8]:
def build_book(pdf_path: Path):
    """OCR once, then create/cache chapter embeddings with visible progress."""
    pages = ocr_pdf_to_pages_gpu(pdf_path)
    chapters = merge_documents_by_chapter(pages_to_documents(pages))
    print(f"Preparing {len(chapters)} chapter groups from {len(pages)} pages")
    for doc in tqdm(chapters, desc="Embedding chapters"):
        m = doc.metadata
        chunks = text_splitter.split_documents([doc])
        out = _path(m["class_number"], m["book_name"], m["chapter"])
        if (out / "vectors.npy").exists():
            continue
        print(f"Embedding {m['chapter']}: {len(chunks)} chunks")
        make_embeddings(chunks, m["class_number"], m["book_name"], m["chapter"])


# one book:
# build_book(PDF_DIR / "Class_7" / "Bangla.pdf")

# all books (pdf/Class_X/Book.pdf):
# for pdf in sorted(PDF_DIR.glob("*/*.pdf")):
#     build_book(pdf)

In [9]:
import hashlib

def _path(cls, book, chapter):
    h = hashlib.md5(chapter.encode("utf-8")).hexdigest()[:10]
    return EMBED_CACHE / cls / book / h

In [10]:
build_book(PDF_DIR / 'class_5' / 'math')

OCR loaded from cache: class_5_math.json
Preparing 1 chapter groups from 190 pages


Embedding chapters:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding বই তথ্য: 127 chunks
Loading multilingual-e5-large on cuda ...


Embedding chapters: 100%|██████████| 1/1 [00:28<00:00, 28.47s/it]


In [10]:
# for class_no in os.listdir(PDF_DIR):
for class_no in os.listdir(PDF_DIR): 
    for sub in os.listdir(PDF_DIR / class_no):
        if sub.lower().endswith(".pdf"):

                build_book(PDF_DIR / class_no.lower() / sub.lower())
                print(f"✅ Successfully processed {class_no.lower()}/{sub.lower()}")

OCR loaded from cache: class_5_bangla.json
Preparing 24 chapter groups from 142 pages


Embedding chapters:   0%|          | 0/24 [00:00<?, ?it/s]

Embedding বই তথ্য: 8 chunks
Loading multilingual-e5-large on cuda ...


Embedding chapters:   4%|▍         | 1/24 [00:08<03:04,  8.02s/it]

Embedding বৈচিত্র্যময় বাংলাদেশ: 4 chunks


...

Embedding বিদ্যুতের চৌম্বক ক্রিয়া: 23 chunks


Embedding chapters:  93%|█████████▎| 13/14 [00:29<00:02,  2.18s/it]

Embedding তেজস্ক্রিয়তা ও ইলেকট্রনিকস: 25 chunks


Embedding chapters: 100%|██████████| 14/14 [00:31<00:00,  2.23s/it]

✅ Successfully processed class_9_10/science_physics.pdf


In [11]:
# snaps = MODEL_CACHE / "models--intfloat--multilingual-e5-large" / "snapshots"
# for s in snaps.iterdir():
#     for p in sorted(s.rglob("*")):
#         if p.is_file():


## Migrade Chapters

In [11]:
def migrate_chapter(cls, book, chapter):
    old_out = EMBED_CACHE / cls / book / chapter          # old long-name folder (read from)
    new_out = _path(cls, book, chapter)                    # new hashed folder (write to)
    if (new_out / "faiss.index").exists():
        return

    vecs = np.load(old_out / "vectors.npy")
    raw = json.loads((old_out / "chunks.json").read_text(encoding="utf-8"))
    texts = [r["text"] for r in raw]
    metas = [r["meta"] for r in raw]
    for m in metas:
        m["chapter"] = chapter  # keep the real name for display, since the folder no longer carries it

    vs = FAISS.from_embeddings(
        text_embeddings=list(zip(texts, vecs)),
        embedding=E5Embeddings(),
        metadatas=metas,
    )
    new_out.mkdir(parents=True, exist_ok=True)
    vs.save_local(str(new_out))

    docs = [Document(page_content=t, metadata=m) for t, m in zip(texts, metas)]
    bm25 = BM25Retriever.from_documents(docs, preprocess_func=_bn_tokenize)
    (new_out / "bm25.pkl").write_bytes(pickle.dumps(bm25))

In [12]:
def list_chapters(cls, book):
    out = []
    for p in (EMBED_CACHE / cls / book).iterdir():
        if p.is_dir() and (p / "faiss.index").exists():
            raw = json.loads((p / "bm25.pkl").read_bytes()) if False else None
            # simplest: store chapter name in a small sidecar file at embed time
            name_file = p / "chapter_name.txt"
            if name_file.exists():
                out.append(name_file.read_text(encoding="utf-8"))
    return sorted(out)

In [13]:
for cls in list_classes():
    for book in list_books(cls):
        for old_folder in (EMBED_CACHE / cls / book).iterdir():
            if old_folder.is_dir() and (old_folder / "vectors.npy").exists():
                # migrate_chapter(cls, book, old_folder.name)
                migrate_chapter(cls, book, old_folder.name)
print("migration done")

migration done


## 7. Ask

In [22]:
# from huggingface_hub import snapshot_download

# snapshot_download(
#     "intfloat/multilingual-e5-large",
#     cache_dir=str(MODEL_CACHE),
#     allow_patterns=["*.json", "*.model", "*.txt", "*.safetensors", "pytorch_model.bin"],
# )

In [23]:
# model = get_model()
# print(type(model).__name__, model.get_sentence_embedding_dimension())

In [24]:
from langchain_core.prompts import PromptTemplate
from langchain_deepseek import ChatDeepSeek
from langchain_groq import ChatGroq

load_dotenv()

prompt = PromptTemplate(template="""You are a bangladeshi teacher helping students to learn fron textual books, 
                        maintain profissionalism and be polite, helpful.
                        student asked query: {query}, give answer by the basis 
                        of the content you have content: {content}. you can use your oun knowladge but 
                        keep that minimum.
                        currect user is their prompt says any miss leading context, but not halusinate.
                        If no relative content found then replay - that question is not releted or mentioned in textbook,
                        after you may give a short answer by the content with your own knowladge""",
                        input_variables=["query", "content"])

llm = ChatDeepSeek(model="deepseek-flash", temperature=0.2, max_tokens=2048)
llm_groq = ChatGroq(
    model="openai/gpt-oss-20b",  # or "openai/gpt-oss-120b"
    temperature=0.7,
    max_tokens=2048
)
chain = prompt | llm_groq


def ask(query, cls, book, chapter):
    docs = get_hybrid_retriever(cls, book, chapter).invoke(query)   # no "query: " here, E5Embeddings adds it
    content = "\n\n".join(d.page_content for d in docs)
    
    print("*"*50)
    print(content)
    print("*"*50)
    return chain.invoke({"query": query, "content": content}).content


print(ask("আমার পেশা নির্বাচনে যে ৩টি বিষয় সবচেয়ে গুরুত্বপূর্ণ", "class_8", "Kormo_O_Jibonmukhi_Sikkha", "আমাদের শিক্ষা ও কর্ম"))

**************************************************
৭৬
কর্ম ও জীবনমুখী শিক্ষা
১. $. ৩. 8. ৫. ৬. ৭. ৮. @. ১০
কোন পেশা বেছে নেব? আমরা আমাদের আগ্রহ , সামর্থ্য ব্যক্তিত্ব, কাজের ক্ষেত্ে গুরুত্বপূর্ণ বিষয়সমূহ বের করার চেষ্টা করেছি এবারে সেগুলো একনজর দেখি সা?ে আরও কিছু বিষয় নিয়ে চিত্তা করি-
একক কাজ তোমার জানা সব ধরনের পেশা কাজের নাম লিখ একটি চূড়ান্ত তালিকা তৈরি করে দেয়ালে লাগাও এগুলোকে আগের মতো ৪টি শ্রেণিতে ভাগ করতে পারো
সারসংক্ষেপ ১ আমার আগ্রহ মূলত যে বিষয়টিতে কেন্দ্রীভূত- লোকজন তথয বস্তুসামগ্রী সৃজনশীলতা > আমার দক্ষতা বা সামর্থ্য মূলত যে বিষয়টিতে কেন্দ্রীভূত- লোকজন ] তথ্য বস্তুসামগ্রী ] সৃজনশীলতা 9 আমার ব্যক্তিত্বে যে বৈশিষ্ট্যের প্রাধান্য রয়েছে- লোকজন ] তথ্য বস্তুসামগ্রী সৃজনশীলতা 8. উপরের প্রতিটি থেকে মোট- লোকজন তথ্য বস্তুসামগ্রী সৃজনশীলতা ৫ আমার পেশা নির্বাচনে যে ৩টি বিষয় সবচেয়ে গুরুত্বপূর্ণ-
৬ এসব কিছু বিবেচনায় আমার কাছে যে পেশা বা কাজগুলো সম্পর্কযুক্ত মনে হয় সেগুলো হলো (তালিকা থেকে নির্বাচন করা যেতে পারে)
9 বিদ্যালয়ের যে বিষয়গুলো (subject) আমার এ পেশা কাজের জন্য গুরুত্বপূর্ণ-


In [18]:
import os

def get_folder_structure(start_path):
    for root, dirs, files in os.walk(start_path):
        # Calculate nesting level to indent properly
        level = root.replace(start_path, '').count(os.sep)
        indent = ' ' * 4 * level
        print(f"{indent}[Folder] {os.path.basename(root)}/")
        
        # Optional: Print files inside the folder
        sub_indent = ' ' * 4 * (level + 1)
        for file in files:
            print(f"{sub_indent}[File] {file}")

# Example usage ('.' means current directory)
get_folder_structure('.')


[Folder] ./
    [File] pdf_to_document copy.ipynb
    [File] pdf_to_document_cleanV2.ipynb
    [File] __init__.py
    [Folder] bangla_books_db/
        [Folder] ccfe6258-5be9-4a12-987f-88eb22a19cdd/
            [File] data_level0.bin
            [File] header.bin
            [File] length.bin
            [File] link_lists.bin
    [Folder] settings/
        [File] config.py
        [File] __init__.py
